# Ingest weather_documents into weather_embeddings using sentence-transformers (Lakebase)

In [0]:
%pip install --upgrade -q 'databricks-sdk>=0.118.0' sentence-transformers pandas psycopg2-binary


In [0]:
dbutils.library.restartPython()

## Config

Widgets let you override the source/destination table names and the
embedding model without editing the notebook - useful when running this
as a scheduled Databricks Job.

In [0]:
# Databricks notebook widgets

dbutils.widgets.text(
    "weather_documents_table",
    "weather_documents",
    "Source table (raw weather documents)",
)
dbutils.widgets.text(
    "weather_embeddings_table",
    "weather_embeddings",
    "Destination table (weather vectors)",
)
dbutils.widgets.text(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2",
    "Embedding model",
)
dbutils.widgets.text(
    "chunk_size",
    "800",
    "Text chunk size (characters)",
)
dbutils.widgets.text(
    "chunk_overlap",
    "100",
    "Chunk overlap (characters)",
)
dbutils.widgets.text(
    "document_limit",
    "500",
    "Maximum unembedded documents per run",
)
dbutils.widgets.text(
    "batch_size",
    "64",
    "Lakebase insert batch size",
)
dbutils.widgets.text(
    "lakebase_secret_scope",
    "database",
    "Lakebase secret scope",
)
dbutils.widgets.text(
    "lakebase_secret_key",
    "lakebase-url",
    "Lakebase URL secret key",
)

LAKEBASE_SECRET_SCOPE = dbutils.widgets.get("lakebase_secret_scope")
LAKEBASE_SECRET_KEY = dbutils.widgets.get("lakebase_secret_key")

WEATHER_DOCUMENTS_TABLE = dbutils.widgets.get("weather_documents_table")
WEATHER_EMBEDDINGS_TABLE = dbutils.widgets.get("weather_embeddings_table")
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model")
CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))
DOCUMENT_LIMIT = int(dbutils.widgets.get("document_limit"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

if CHUNK_SIZE <= 0:
    raise ValueError("chunk_size must be greater than 0")

if CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("chunk_overlap must be at least 0 and smaller than chunk_size")

if DOCUMENT_LIMIT <= 0 or BATCH_SIZE <= 0:
    raise ValueError("document_limit and batch_size must be greater than 0")

# The homework requires this exact model. Its vectors always have 384 values,
# so the Lakebase pgvector column must be VECTOR(384).
if EMBEDDING_MODEL_NAME != "sentence-transformers/all-MiniLM-L6-v2":
    raise ValueError(
        "This weather pipeline requires "
        "'sentence-transformers/all-MiniLM-L6-v2'."
    )

EMBEDDING_DIM = 384

print(
    f"Documents: {WEATHER_DOCUMENTS_TABLE}\n"
    f"Embeddings: {WEATHER_EMBEDDINGS_TABLE}\n"
    f"Model: {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIM} dimensions)\n"
    f"Chunks: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars\n"
    f"Document limit: {DOCUMENT_LIMIT}, DB batch size: {BATCH_SIZE}"
)

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def get_lakebase_url() -> str:
    """Read and decode the Lakebase PostgreSQL URL from Databricks Secrets."""
    secret = w.secrets.get_secret(
        scope=LAKEBASE_SECRET_SCOPE,
        key=LAKEBASE_SECRET_KEY,
    )
    return base64.b64decode(secret.value).decode("utf-8")


lakebase_url = get_lakebase_url()
parsed = urlparse(lakebase_url)

db_host = parsed.hostname
db_port = parsed.port or 5432
db_name = parsed.path.lstrip("/")
db_user = parsed.username
db_password = parsed.password

print("Lakebase connection configured:")
print(f"  Host: {db_host}:{db_port}")
print(f"  Database: {db_name}")
print(f"  User: {db_user}")

In [0]:
import traceback

import psycopg2
from psycopg2.extras import RealDictCursor

print(f"Testing Lakebase connection: {db_host}:{db_port}/{db_name}")

try:
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode="require",
        connect_timeout=10,
        cursor_factory=RealDictCursor,
    )

    with conn.cursor() as cursor:
        cursor.execute(
            f"SELECT COUNT(*) AS count FROM {WEATHER_DOCUMENTS_TABLE}"
        )
        count = cursor.fetchone()["count"]

        print(
            f"✅ Connection successful! "
            f"Found {count} rows in {WEATHER_DOCUMENTS_TABLE}."
        )

        cursor.execute(
            f"""
            SELECT id, location, source_type, headline, synced_at
            FROM {WEATHER_DOCUMENTS_TABLE}
            ORDER BY synced_at DESC
            LIMIT 5
            """
        )
        rows = cursor.fetchall()

        print("\nLatest weather documents:")
        for row in rows:
            print(dict(row))

    conn.close()
    print("\n✅ psycopg2 connection works correctly.")

except Exception as exc:
    print(f"❌ Connection failed: {exc}")
    traceback.print_exc()

## Database Setup Instructions

Before running this notebook, you must manually create the required tables
in your Lakebase Postgres database


## Load unembedded weather documents

Reads normalized alerts and forecasts from `weather_documents`.
The text to embed is `narrative_text`. Documents already embedded with
the selected model are skipped.

In [0]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require",
)

try:
    query = f"""
        SELECT
            d.id,
            d.location,
            d.source_type,
            d.headline,
            d.narrative_text,
            d.issued_at,
            d.effective_at,
            d.synced_at,
            d.narrative_text AS embedding_text
        FROM {WEATHER_DOCUMENTS_TABLE} d
        WHERE TRIM(COALESCE(d.narrative_text, '')) != ''
          AND NOT EXISTS (
              SELECT 1
              FROM {WEATHER_EMBEDDINGS_TABLE} e
              WHERE e.document_id = d.id
                AND e.model_name = %s
          )
        ORDER BY d.synced_at ASC
        LIMIT %s
    """

    weather_df = pd.read_sql_query(
        query,
        conn,
        params=(EMBEDDING_MODEL_NAME, DOCUMENT_LIMIT),
    )

    print(
        f"Loaded {len(weather_df)} unembedded weather documents "
        f"from {WEATHER_DOCUMENTS_TABLE}"
    )
    display(weather_df.head(5))

finally:
    conn.close()

## Compute embeddings

Loads the embedding model once, splits long `narrative_text` values
into overlapping chunks, then creates a 384-dimensional vector per chunk.

In [0]:
import hashlib
import os

import pandas as pd
from sentence_transformers import SentenceTransformer

os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"


def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """Split text into overlapping character chunks."""
    text = (text or "").strip()

    if not text:
        return []

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(text):
            break

    return chunks


def make_embedding_id(document_id: str, chunk_index: int) -> str:
    """Stable ID: the same document/chunk/model never creates a duplicate."""
    value = f"{document_id}:{chunk_index}:{EMBEDDING_MODEL_NAME}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


# Turn one weather document into one or more chunk rows.
chunk_rows = []

for _, document in weather_df.iterrows():
    chunks = chunk_text(document["embedding_text"])

    for chunk_index, chunk in enumerate(chunks):
        chunk_rows.append(
            {
                "id": make_embedding_id(document["id"], chunk_index),
                "document_id": document["id"],
                "location": document["location"],
                "source_type": document["source_type"],
                "headline": document["headline"],
                "chunk_index": chunk_index,
                "chunk_text": chunk,
            }
        )

chunks_df = pd.DataFrame(chunk_rows)

print(
    f"Created {len(chunks_df)} chunks from "
    f"{len(weather_df)} weather documents."
)

if chunks_df.empty:
    print("No unembedded weather text to process.")
    weather_embeddings_df = pd.DataFrame()
else:
    print(f"Loading embedding model {EMBEDDING_MODEL_NAME}...")
    model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        cache_folder="/tmp/.cache/huggingface",
    )

    print("Computing embeddings...")
    all_embeddings = []

    for start in range(0, len(chunks_df), BATCH_SIZE):
        batch = chunks_df.iloc[start:start + BATCH_SIZE]

        vectors = model.encode(
            batch["chunk_text"].tolist(),
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_embeddings.extend(vectors.tolist())

        print(
            f"Processed {min(start + BATCH_SIZE, len(chunks_df))}"
            f"/{len(chunks_df)} chunks"
        )

    weather_embeddings_df = chunks_df.copy()
    weather_embeddings_df["embedding"] = all_embeddings
    weather_embeddings_df["model_name"] = EMBEDDING_MODEL_NAME

    print(
        f"Computed {len(weather_embeddings_df)} embeddings "
        f"using {EMBEDDING_MODEL_NAME}."
    )

    display(weather_embeddings_df.head(5))

## Ensure the pgvector destination table exists

The `pgvector` extension must be enabled and the destination table
created with the correct vector dimension before inserting embeddings.

In [0]:
print(f"Required embedding dimension: {EMBEDDING_DIM}")
print(f"Documents table: {WEATHER_DOCUMENTS_TABLE}")
print(f"Embeddings table: {WEATHER_EMBEDDINGS_TABLE}")
print("\nRequired schema: embedding VECTOR(384)")
print(
    "Run sql/02_setup_weather_embeddings.sql, "
    "or use lakebase.ensure_weather_embeddings_table()."
)

## Upsert embeddings into Lakebase

Written in batches via psycopg2's `executemany` for throughput.
Each embedding is cast to Postgres' `vector` type via `::vector`.

In [0]:
import psycopg2
from datetime import UTC, datetime

from psycopg2.extras import execute_values

if not weather_embeddings_df.empty:
    print(
        f"Inserting {len(weather_embeddings_df)} embeddings "
        f"into {WEATHER_EMBEDDINGS_TABLE}..."
    )

    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode="require",
    )

    try:
        with conn.cursor() as cursor:
            created_at = datetime.now(UTC)

            # pgvector expects text in this format: [0.12,-0.44,...]
            insert_data = [
                (
                    row["id"],
                    row["document_id"],
                    int(row["chunk_index"]),
                    row["chunk_text"],
                    "[" + ",".join(
                        str(float(value))
                        for value in row["embedding"]
                    ) + "]",
                    row["model_name"],
                    created_at,
                )
                for _, row in weather_embeddings_df.iterrows()
            ]

            insert_sql = f"""
                INSERT INTO {WEATHER_EMBEDDINGS_TABLE} (
                    id,
                    document_id,
                    chunk_index,
                    chunk_text,
                    embedding,
                    model_name,
                    created_at
                )
                VALUES %s
                ON CONFLICT (id) DO UPDATE
                SET
                    chunk_text = EXCLUDED.chunk_text,
                    embedding = EXCLUDED.embedding,
                    model_name = EXCLUDED.model_name,
                    created_at = EXCLUDED.created_at
            """

            execute_values(
                cursor,
                insert_sql,
                insert_data,
                template="(%s, %s, %s, %s, %s::vector, %s, %s)",
                page_size=BATCH_SIZE,
            )

        conn.commit()
        print(f"✅ Inserted/upserted {len(insert_data)} weather embeddings.")

    finally:
        conn.close()

else:
    print("No weather embeddings to write.")

## HNSW Index Performance Benchmark

Now that we have embeddings, let's benchmark the performance difference between vector search **with** vs **without** the HNSW index.

This will:
1. Run queries WITH the HNSW index
2. Drop the index
3. Run queries WITHOUT the index (sequential scan)
4. Recreate the index
5. Report performance comparison

⚠️ **Note:** This temporarily drops and recreates the index.

In [0]:
import time

BENCHMARK_QUERIES = [
    "tornado warning severe thunderstorm",
    "flash flood risk near rivers",
    "winter storm snow accumulation",
    "heat advisory high temperatures",
    "hurricane tropical storm winds",
]

BENCHMARK_TOP_K = 10
BENCHMARK_ITERATIONS = 5

print(f"Benchmark config: {len(BENCHMARK_QUERIES)} queries, {BENCHMARK_ITERATIONS} iterations each")

In [0]:
def embed_query(query: str) -> str:
    """Embed query and format for pgvector."""
    embedding = model.encode(query, normalize_embeddings=True).tolist()
    return "[" + ",".join(str(float(v)) for v in embedding) + "]"

def run_benchmark_queries(queries, iterations):
    """Run benchmark and return timing results."""
    search_sql = """
        SELECT d.id, d.location, d.headline, e.chunk_text,
               1 - (e.embedding <=> %s::vector) AS similarity
        FROM weather_embeddings e
        JOIN weather_documents d ON d.id = e.document_id
        ORDER BY e.embedding <=> %s::vector
        LIMIT %s
    """
    results = {}
    conn = psycopg2.connect(
        host=db_host, port=db_port, dbname=db_name,
        user=db_user, password=db_password, sslmode="require"
    )
    try:
        with conn.cursor() as cur:
            for query in queries:
                query_vector = embed_query(query)
                times = []
                for _ in range(iterations):
                    start = time.perf_counter()
                    cur.execute(search_sql, (query_vector, query_vector, BENCHMARK_TOP_K))
                    _ = cur.fetchall()
                    times.append((time.perf_counter() - start) * 1000)
                results[query] = times
    finally:
        conn.close()
    return results

print("[Phase 1] Running WITH HNSW index...")
with_index_results = run_benchmark_queries(BENCHMARK_QUERIES, BENCHMARK_ITERATIONS)
for query, times in with_index_results.items():
    print(f"{query}: {sum(times)/len(times):.2f} ms avg")
print("✅ Phase 1 complete")

In [0]:
conn = psycopg2.connect(
    host=db_host, port=db_port, dbname=db_name,
    user=db_user, password=db_password, sslmode="require"
)
try:
    with conn.cursor() as cur:
        print("[Phase 2] Dropping HNSW index...")
        cur.execute("DROP INDEX IF EXISTS idx_weather_embeddings_hnsw")
        conn.commit()
        print("✅ Index dropped\n")
finally:
    conn.close()

print("[Phase 3] Running WITHOUT index (sequential scan)...")
without_index_results = run_benchmark_queries(BENCHMARK_QUERIES, BENCHMARK_ITERATIONS)
for query, times in without_index_results.items():
    print(f"{query}: {sum(times)/len(times):.2f} ms avg")
print("✅ Phase 3 complete")

In [0]:
conn = psycopg2.connect(
    host=db_host, port=db_port, dbname=db_name,
    user=db_user, password=db_password, sslmode="require"
)
index_start = time.perf_counter()
try:
    with conn.cursor() as cur:
        print("[Phase 4] Recreating HNSW index...")
        cur.execute("""
            CREATE INDEX idx_weather_embeddings_hnsw
            ON weather_embeddings
            USING hnsw (embedding vector_cosine_ops)
        """)
        conn.commit()
finally:
    conn.close()
index_time = (time.perf_counter() - index_start) * 1000
print(f"✅ Index recreated in {index_time:.2f} ms\n")

# Calculate overall stats
all_with = [t for times in with_index_results.values() for t in times]
all_without = [t for times in without_index_results.values() for t in times]
avg_with = sum(all_with) / len(all_with)
avg_without = sum(all_without) / len(all_without)
speedup = avg_without / avg_with
improvement = ((avg_without - avg_with) / avg_without) * 100

print("="*70)
print("BENCHMARK RESULTS")
print("="*70)
print(f"Total queries: {len(all_with)}")
print(f"WITH index:    {avg_with:>7.2f} ms avg")
print(f"WITHOUT index: {avg_without:>7.2f} ms avg")
print(f"🚀 Speedup:    {speedup:>7.2f}x ({improvement:.1f}% faster)")
print(f"Index creation: {index_time:.2f} ms")
print("="*70)